In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

In [3]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [4]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
test_df

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...
...,...,...,...,...,...,...,...
495,496,What are the constituents of cold dark matter?,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.
496,497,Pick the best possible answer: What is a plane...,A framework of planets that are all located in...,A mechanism of planets that are all the same s...,Any set of gravitationally bound non-stellar o...,A mechanism of planets that are all located in...,A structure of planets that are all made of gas.
497,498,Pick the best possible answer: What is magneti...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...
498,499,Determine the correct option: What is the evid...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The star S2 follows an elliptical orbit with a...


In [6]:
# frequency distribution of answers
answer_counts = train_df['answer'].value_counts()
print(answer_counts)


# most and least frequent option 
most_frequent = answer_counts.max()
least_frequent = answer_counts.min()

# sum
sum_most_least = most_frequent + least_frequent
print(f"Sum of most and least frequent options: {sum_most_least}")

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Sum of most and least frequent options: 814


In [7]:
import string
cleaned_prompts = train_df['prompt'].str.lower().str.translate(str.maketrans('', '', string.punctuation))
words_series = cleaned_prompts.str.split().explode()

vocab_size = words_series.dropna().nunique()

print(f"vocabulary size: {vocab_size}")

vocabulary size: 859


In [8]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
cleaned_prompt = cleaned_prompts.iloc[0]
words = cleaned_prompt.split()

filtered_words = [word for word in words if word not in ENGLISH_STOP_WORDS]
remaining_count = len(filtered_words)

print(f"Word count after removing stop words: {remaining_count}")

Word count after removing stop words: 13


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

# .fillna('') prevents errors from missing data
# .agg(' '.join, axis=1) joins the text across columns with a space
corpus = train_df[text_cols].fillna('').astype(str).agg(' '.join, axis=1).tolist()

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)

num_features = tfidf_matrix.shape[1]
num_features

2762

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

prompt_text = train_df.loc[train_df['id'] == 1, 'prompt'].values[0]
option_A_text = train_df.loc[train_df['id'] == 1, 'A'].values[0]


prompt_vector = vectorizer.transform([prompt_text])
option_A_vector = vectorizer.transform([option_A_text])

similarity_matrix = cosine_similarity(prompt_vector, option_A_vector)
similarity_score = similarity_matrix[0][0]
similarity_score

np.float64(0.27202429519891635)

In [11]:
options = ['A', 'B', 'C', 'D', 'E']
correct_predictions = 0
total_rows = len(train_df)

for index, row in train_df.iterrows():
    
    prompt_vector = vectorizer.transform([str(row['prompt'])])
    
    highest_sim_score = -1
    predicted_option = None
    
    for opt in options:
        option_vector = vectorizer.transform([str(row[opt])])
        sim_score = cosine_similarity(prompt_vector, option_vector)[0][0]
        
        if sim_score > highest_sim_score:
            highest_sim_score = sim_score
            predicted_option = opt
            
    if predicted_option == row['answer']:
        correct_predictions += 1

accuracy_percentage = (correct_predictions / total_rows) * 100

print(f"Correctly Matched by Cosine Similarity: {correct_predictions}")
print(f"Accuracy Percentage: {accuracy_percentage:.2f}%")

Correctly Matched by Cosine Similarity: 271
Accuracy Percentage: 13.55%
